# Checkpoint 1: Sentinel-2 Data Acquisition
**Team:** MozwiloFortune  
**Region:** Po Valley, Italy (Tile T32TMR)  
**Period:** March-October 2018

## Requirements
- ✅ Area: EU (not Iceland) - Po Valley, Italy
- ✅ Level-2A products only
- ✅ Cloud coverage ≤ 30%
- ✅ 4 acquisitions (Mar/Jun/Sep/Oct 2018)
- ✅ Stored in: `/p/scratch/training2600/teamMozwiloFortune/data/`

## Setup

In [ ]:
import requests
import os
import json
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt

# Copernicus credentials
COPERNICUS_CLIENT_ID = os.getenv('COPERNICUS_CLIENT_ID', 'sh-13513c84-f085-40da-814b-8b205c501135')
COPERNICUS_CLIENT_SECRET = os.getenv('COPERNICUS_CLIENT_SECRET', 'OVKVy3k3b1Rc96PtwrM1D08ARBbNd4YL')

# API endpoints
AUTH_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
SEARCH_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
DOWNLOAD_URL = "https://zipper.dataspace.copernicus.eu/odata/v1/Products"

print("✓ Setup complete")

## Authentication

In [ ]:
def get_access_token(client_id, client_secret):
    """Get OAuth2 access token from Copernicus Dataspace."""
    data = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret
    }
    
    try:
        response = requests.post(AUTH_URL, data=data, timeout=30)
        response.raise_for_status()
        token_data = response.json()
        return token_data.get('access_token')
    except Exception as e:
        print(f"Authentication failed: {e}")
        return None

# Get token
print("Authenticating...")
access_token = get_access_token(COPERNICUS_CLIENT_ID, COPERNICUS_CLIENT_SECRET)
if access_token:
    print("✓ Authentication successful")
else:
    print("✗ Authentication failed")

## Downloaded Products

Our 4 temporal acquisitions for Po Valley (T32TMR):

In [ ]:
# Data directory
DATA_DIR = Path("/p/scratch/training2600/teamMozwiloFortune/data/sentinel2_extracted")

# Our 4 acquisitions
ACQUISITIONS = [
    "S2B_MSIL2A_20180308T102019_N0500_R065_T32TMR_20230910T101040.SAFE",  # March
    "S2B_MSIL2A_20180616T102019_N0500_R065_T32TMR_20230715T133909.SAFE",  # June
    "S2B_MSIL2A_20180904T102019_N0500_R065_T32TMR_20230824T235353.SAFE",  # September
    "S2B_MSIL2A_20181024T102059_N0500_R065_T32TMR_20230816T103450.SAFE",  # October
]

print("Verifying acquisitions...\n")
for i, safe in enumerate(ACQUISITIONS, 1):
    path = DATA_DIR / safe
    if path.exists():
        # Extract date from filename
        date_str = safe.split('_')[2][:8]
        date = datetime.strptime(date_str, '%Y%m%d').strftime('%B %d, %Y')
        print(f"✓ Acquisition {i}: {date}")
        print(f"  {safe}")
        print()
    else:
        print(f"✗ MISSING: {safe}")
        print()

print("\n✅ All 4 acquisitions verified!")

## Visualization (March 2018 RGB)

In [ ]:
import rasterio
import numpy as np

# Visualize first acquisition (March)
safe_path = DATA_DIR / ACQUISITIONS[0]

# Find band files
def find_band(safe_path, band):
    pattern = f"GRANULE/*/IMG_DATA/R10m/*_{band}_10m.jp2"
    matches = list(safe_path.glob(pattern))
    return str(matches[0]) if matches else None

# Read RGB bands
r_path = find_band(safe_path, 'B04')  # Red
g_path = find_band(safe_path, 'B03')  # Green
b_path = find_band(safe_path, 'B02')  # Blue

if all([r_path, g_path, b_path]):
    with rasterio.open(r_path) as src:
        r = src.read(1)
    with rasterio.open(g_path) as src:
        g = src.read(1)
    with rasterio.open(b_path) as src:
        b = src.read(1)
    
    # Stack and normalize for display
    rgb = np.dstack([r, g, b])
    
    # Percentile stretch for visualization
    p2, p98 = np.percentile(rgb[rgb > 0], [2, 98])
    rgb_norm = np.clip((rgb - p2) / (p98 - p2), 0, 1)
    
    # Plot
    plt.figure(figsize=(12, 10))
    plt.imshow(rgb_norm)
    plt.title('Po Valley, Italy - March 2018 (RGB)', fontsize=14, fontweight='bold')
    plt.xlabel('East →', fontsize=12)
    plt.ylabel('North ↑', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    print(f"Image size: {r.shape[1]} × {r.shape[0]} pixels")
    print(f"Coverage: ~100km × 100km")
else:
    print("✗ Could not find all RGB bands")

## Summary

**✅ Checkpoint 1 Complete**

- Region: Po Valley, Italy (Tile T32TMR)
- Products: 4 Sentinel-2 Level-2A acquisitions
- Period: March-October 2018
- Cloud coverage: < 30%
- Storage: `/p/scratch/training2600/teamMozwiloFortune/data/`

**Next:** Checkpoint 2 - Extract training patches with CORINE labels